# Curriculum 03 · Lab 2 — Chroma: the persistent vector store

**Goal:** Build a Chroma store over the same rag-mini-wikipedia subset as lab
01 and prove persistence the hard way — the store is closed, reopened **in a
brand-new Python process**, and answers the same question with the same
passage and the same score. FAISS's index (lab 01) vanishes at process exit;
Chroma's lives in a directory on disk.

```
Persistence : a directory — chroma.sqlite3 (registry) + per-collection HNSW files
Score       : l2 space — the SAME squared-L2 numbers FAISS reports (lab 01)
Reopening   : load() — never add() again (re-adding UPSERTS and duplicates)
Proof       : close -> reopen in-process AND in a brand-new Python process
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Data        : rag-mini-wikipedia (first 100 passages, questions 1606/1610)
```

**Why persistence:** "the store keeps your data" is the property that
separates a demo from an application. This lab makes it concrete: it prints
the on-disk layout and file sizes, then deletes the first Chroma instance
and asks the same question again from a fresh process reading only the
directory — if the data lived in RAM, the second ask would fail.

This is the second lab of track 03-vector-databases (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
imports pandas plus the repo's vector-store classes and BGE embedder, and
puts the repo-root component library on `sys.path` so this notebook reuses
`src/vectordb/*.py` and `src/embeddings/bge.py` exactly like the lab script.

**WHY:** Everything embeds **locally** with BGE via sentence-transformers —
no API embeddings anywhere. The store classes live in the repo's shared
component library (`src/vectordb/`), not inside the lab, so the exact same code
path runs here, in the `.py`, and in later tracks.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script) or from
the notebook's own folder (the Jupyter default) — and `cd`s into it so every
path stays repo-relative.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model is loaded lazily
when the experiment cell first calls it.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings
#   faiss-cpu             -> the lab-01 FAISS score baseline
#   chromadb              -> the persistent Chroma store
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu chromadb pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import gc
import json
import os
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from embeddings.bge import BGEEmbedding  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from vectordb.chroma import ChromaVectorStore  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** Same corpus/embedding constants as lab 01 plus the
Chroma-specific ones: `SCORE_TOL` (FAISS vs Chroma are separate float32 code
paths, so the two scores are allowed last-digit jitter) and
`COLLECTION = "lab02"` (the collection name inside the persistent directory).

**WHY:** `QUESTION_IDS = [1606, 1610]` — two questions are enough because
the lab's real subject is persistence, not ranking; the ranking checks
against FAISS are the same ones lab 01 already locked in.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus
QUESTION_IDS = [1606, 1610]  # real questions; answers live inside the subset
TOP_K = 3
PREVIEW = 62
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DIM = 768
SCORE_TOL = 1e-3  # FAISS vs Chroma are separate float32 code paths; allow last-digit jitter
COLLECTION = "lab02"


## 2 · Load — corpus + questions (same helpers as lab 01)

**WHAT:** `load_passages` pulls the first `n` passages (text + ids) from
`passages.parquet`; `load_questions` pulls specific rows by id from
`test.parquet`; `preview` flattens a passage for one-line printing.

**WHY:** Identical helpers to lab 01 keep the two labs directly comparable —
the only thing that changes between them is the store.


In [4]:
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3 · Experiment — embed once, feed the same vectors to both stores,
then prove Chroma persists across a process boundary

**WHAT:** `run_experiment` embeds the subset once, indexes the *same*
vectors into FAISS (the lab-01 baseline) and into a persistent Chroma
collection, then does the persistence proof in two steps: an in-process
reopen (drop the first instance, `load()` a fresh one, no re-add) and a
cross-process reopen (a brand-new Python interpreter, launched via
subprocess, reads the same directory and answers Q1610).

**WHY:** The in-process reopen proves the data is on disk, not pinned in the
first instance's objects; the cross-process reopen proves it is not even
pinned in this Python interpreter. The subprocess is the strongest possible
"persistence" claim — nothing from this notebook survives into it except the
directory path.


In [5]:
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]

    # --- Embed the subset once; BOTH stores index the same vectors ----------
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME)
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0
    query_vecs = [embedder.embed_query(q) for _, q in questions]

    # --- FAISS (in-memory, lab 01) — score cross-check baseline ------------
    faiss_store = FAISSVectorStore()
    faiss_store.add(chunks, embeddings=passage_vecs)
    faiss_scored = [faiss_store.query_with_scores(q, top_k=TOP_K) for q in query_vecs]

    # --- Chroma (persistent) — added once, then reopened twice --------------
    tmp = tempfile.TemporaryDirectory(prefix="lab02_chroma_")
    persist_dir = tmp.name

    chroma_store = ChromaVectorStore(collection_name=COLLECTION, persist_dir=persist_dir)
    t0 = time.perf_counter()
    chroma_store.add(chunks, embeddings=passage_vecs)
    add_s = time.perf_counter() - t0
    chroma_scored = [chroma_store.query_with_scores(q, top_k=TOP_K) for q in query_vecs]

    # On-disk artifact listing (proves the data lives outside RAM).
    disk_files = sorted(os.listdir(persist_dir))
    disk_bytes = {
        name: os.path.getsize(os.path.join(persist_dir, name)) for name in disk_files
    }

    # In-process reopen: drop the first instance, open a fresh one, no re-add.
    del chroma_store
    gc.collect()
    reopened = ChromaVectorStore(collection_name=COLLECTION, persist_dir=persist_dir)
    reopened.load()
    reopened_scored = [reopened.query_with_scores(q, top_k=TOP_K) for q in query_vecs]

    # Cross-process reopen: a brand-new Python interpreter reads the same dir.
    qvec_file = os.path.join(persist_dir, "query_vector.json")
    with open(qvec_file, "w") as fh:
        json.dump(query_vecs[1], fh)  # Q1610 "Who founded Montevideo?"
    sub_script = (
        "import sys, json\n"
        f"sys.path.insert(0, {str(REPO_ROOT / 'src')!r})\n"
        "from vectordb.chroma import ChromaVectorStore\n"
        f"q = json.load(open({qvec_file!r}))\n"
        f"s = ChromaVectorStore(collection_name={COLLECTION!r}, persist_dir={persist_dir!r})\n"
        "s.load()\n"
        "doc, score = s.query_with_scores(q, 1)[0]\n"
        'print(f"{doc.metadata.get(chr(105)+chr(100), chr(63))}|{score:.4f}")\n'
    )
    t0 = time.perf_counter()
    proc = subprocess.run(
        [sys.executable, "-c", sub_script], capture_output=True, text=True, timeout=120
    )
    reopen_s = time.perf_counter() - t0
    if proc.returncode != 0:
        raise RuntimeError(f"reopen subprocess failed: {proc.stderr[-500:]}")
    sub_pid_str, sub_score_str = proc.stdout.strip().split("|")

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "embed_s": embed_s,
        "add_s": add_s,
        "reopen_s": reopen_s,
        "faiss_scored": faiss_scored,
        "chroma_scored": chroma_scored,
        "reopened_scored": reopened_scored,
        "disk_files": disk_files,
        "disk_bytes": disk_bytes,
        "sub_pid": int(sub_pid_str),
        "sub_score": float(sub_score_str),
        "dim": len(passage_vecs[0]),
        "indexed": len(passage_vecs),
        "persist_dir": persist_dir,
        "tmp": tmp,
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — this embeds the corpus, builds both
stores, and runs both reopen proofs (the subprocess step takes a few
seconds). The artifact dict is kept as `exp`.

**WHY:** As in lab 01, demo and gate both read this single `exp`, so the
printed numbers and the verified numbers come from the same run.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the on-disk layout of the persistent directory
(with real file sizes), the FAISS-vs-Chroma top-3 side by side (the `SAME`
labels), and the two persistence proofs — the in-process reopen scores and
the brand-new-process answer.

**WHY:** Look at the two score columns first: they match to the last few
digits because both stores default to the l2 space — same vectors, same
ranking, same numbers, one in RAM and one on disk. Then read the reopen
lines: the data survived because it lives in `chroma.sqlite3` + the HNSW
files, not in RAM.


In [7]:
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 02 — Chroma: the persistent vector store")
    print(f"{BGE_MODEL_NAME} | l2 space | persist_dir on disk")
    print("=" * 66)

    print(f"\n[1] Corpus + embedding:")
    print(f"    {exp['indexed']} passages, dim {exp['dim']}, embedded in {exp['embed_s']:.2f}s")
    print(f"    same vectors indexed into FAISS (lab 01 baseline) AND Chroma")

    print(f"\n[2] Chroma index build:")
    print(f"    {exp['indexed']} passages added in {exp['add_s']:.3f}s")
    print("    on-disk layout of persist_dir:")
    for name, size in exp["disk_bytes"].items():
        print(f"      {name:<45} {size:>9,} B")
    print(f"      {'total':<45} {sum(exp['disk_bytes'].values()):>9,} B")

    print(f"\n[3] Top-{TOP_K} per question — FAISS vs Chroma (squared L2, LOWER = better):")
    for i, (qid, qtext) in enumerate(exp["questions"]):
        print(f'\n    Q[{qid}] "{qtext}"')
        for (fdoc, fscore), (cdoc, cscore) in zip(
            exp["faiss_scored"][i], exp["chroma_scored"][i]
        ):
            match = "SAME" if fdoc.metadata["id"] == cdoc.metadata["id"] else "DIFF"
            print(f"      faiss {fscore:8.4f}  chroma {cscore:8.4f}  "
                  f"[passage {cdoc.metadata['id']}] {preview(cdoc.page_content)}  {match}")

    print(f"\n[4] Persistence — close, reopen, ask again:")
    print("    same top-1 after in-process reopen:")
    for (qid, _), hits in zip(exp["questions"], exp["reopened_scored"]):
        doc, score = hits[0]
        print(f"      Q[{qid}] -> [passage {doc.metadata['id']}] score {score:.4f}")
    print(f"    brand-new Python process reopened the same dir in {exp['reopen_s']:.2f}s:")
    print(f"      Q[{exp['questions'][1][0]}] -> [passage {exp['sub_pid']}] "
          f"score {exp['sub_score']:.4f}")
    print("    the first Chroma instance was deleted; the data survived because")
    print("    it lives in chroma.sqlite3 + the HNSW files, not in RAM.")

    print("\n[5] Takeaway")
    print("    Chroma swaps lab 01's 'index vanishes at exit' for a directory")
    print("    on disk. The price is visible in [2]: building the index touches")
    print("    sqlite and the HNSW files, so add() is slower than FAISS's.")
    print("    And reopening must call load(), never add() again — re-adding")
    print("    upserts and duplicates every passage in the existing collection.")


In [8]:
print_demo(exp)


Lab 02 — Chroma: the persistent vector store
BAAI/bge-base-en-v1.5 | l2 space | persist_dir on disk

[1] Corpus + embedding:
    100 passages, dim 768, embedded in 15.02s
    same vectors indexed into FAISS (lab 01 baseline) AND Chroma

[2] Chroma index build:
    100 passages added in 2.968s
    on-disk layout of persist_dir:
      5a828a20-56a6-4bc5-8c3c-808956e01457                120 B
      chroma.sqlite3                                  925,696 B
      total                                           925,816 B

[3] Top-3 per question — FAISS vs Chroma (squared L2, LOWER = better):

    Q[1606] "Is Uruguay's capital Montevideo?"
      faiss   0.2650  chroma   0.2650  [passage 36] Montevideo, Uruguay's capital.  SAME
      faiss   0.4931  chroma   0.4931  [passage 15] Uruguay's capital, Montevideo, was founded by the Spanish in t...  SAME
      faiss   0.5141  chroma   0.5141  [passage 64] Montevideo, capital of the country. A view of pedestrian stree...  SAME

    Q[1610] "Who foun

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate` the lab script runs with `--verify`:
dimension/count, FAISS-vs-Chroma ranking and score agreement, the
`chroma.sqlite3` disk check, both content checks, and both reopen checks
(in-process ids, and the new-process id + score).

**WHY:** `python 02-chroma-persistent.py --verify` must print 10/10 PASS;
this cell proves the notebook reproduces the verified `.py` exactly.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == BGE_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Chroma's score convention matches FAISS's (both squared L2, lower = better).
    all_same_rank = True
    all_scores_close = True
    for fhits, chits in zip(exp["faiss_scored"], exp["chroma_scored"]):
        for (fdoc, fscore), (cdoc, cscore) in zip(fhits, chits):
            all_same_rank &= fdoc.metadata["id"] == cdoc.metadata["id"]
            all_scores_close &= abs(fscore - cscore) < SCORE_TOL
    checks.append(("FAISS and Chroma return the same passages in the same order", all_same_rank))
    checks.append(("FAISS and Chroma scores agree within tolerance", all_scores_close))

    # The store actually wrote to disk.
    checks.append(("persist_dir contains chroma.sqlite3", "chroma.sqlite3" in exp["disk_files"]))

    # Content checks (same as lab 01 — the top-1 answer passages).
    q1610_top = exp["chroma_scored"][1][0][0].page_content.lower()
    checks.append(("Q1610 top-1 names the Spanish founder of Montevideo", "spanish" in q1610_top))
    q1606_top = exp["chroma_scored"][0][0][0].page_content.lower()
    checks.append(("Q1606 top-1 mentions Montevideo", "montevideo" in q1606_top))

    # In-process reopen reproduces the original ranking exactly.
    reopen_matches = all(
        o[0][0].metadata["id"] == r[0][0].metadata["id"]
        for o, r in zip(exp["chroma_scored"], exp["reopened_scored"])
    )
    checks.append(("in-process reopen reproduces every top-1 passage id", reopen_matches))

    # Cross-process reopen answers Q1610 with the same passage and score.
    orig = exp["chroma_scored"][1][0]
    checks.append(("new-process reopen matches the original Q1610 top-1 id",
                   exp["sub_pid"] == orig[0].metadata["id"]))
    checks.append(("new-process reopen matches the original Q1610 score",
                   abs(exp["sub_score"] - orig[1]) < SCORE_TOL))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


if __name__ == "__main__":
    exp = run_experiment()
    try:
        if "--verify" in sys.argv:
            sys.exit(verify_gate(exp))
        print_demo(exp)
    finally:
        exp["tmp"].cleanup()  # remove the persistent dir after the demo/gate


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Lab 02 — Chroma: the persistent vector store
BAAI/bge-base-en-v1.5 | l2 space | persist_dir on disk

[1] Corpus + embedding:
    100 passages, dim 768, embedded in 6.45s
    same vectors indexed into FAISS (lab 01 baseline) AND Chroma

[2] Chroma index build:
    100 passages added in 0.073s
    on-disk layout of persist_dir:
      74fe6f5b-e1f3-4ac3-8fb0-c546da634e7d                120 B
      chroma.sqlite3                                  925,696 B
      total                                           925,816 B

[3] Top-3 per question — FAISS vs Chroma (squared L2, LOWER = better):

    Q[1606] "Is Uruguay's capital Montevideo?"
      faiss   0.2650  chroma   0.2650  [passage 36] Montevideo, Uruguay's capital.  SAME
      faiss   0.4931  chroma   0.4931  [passage 15] Uruguay's capital, Montevideo, was founded by the Spanish in t...  SAME
      faiss   0.5141  chroma   0.5141  [passage 64] Montevideo, capital of the country. A view of pedestrian stree...  SAME

    Q[1610] "Who found

In [10]:
verify_gate(exp)


verification gate:
  [PASS] embedding dimension is 768 (BGE base)
  [PASS] exactly 100 passages indexed
  [PASS] FAISS and Chroma return the same passages in the same order
  [PASS] FAISS and Chroma scores agree within tolerance
  [PASS] persist_dir contains chroma.sqlite3
  [PASS] Q1610 top-1 names the Spanish founder of Montevideo
  [PASS] Q1606 top-1 mentions Montevideo
  [PASS] in-process reopen reproduces every top-1 passage id
  [PASS] new-process reopen matches the original Q1610 top-1 id
  [PASS] new-process reopen matches the original Q1610 score


0

## 7 · Cleanup — remove the persistent directory

**WHAT:** Deletes the temporary Chroma directory the experiment wrote, the
same way the lab script's `finally` block does.

**WHY:** The directory is a regenerable artifact — keeping it would leak a
few MB per run into /tmp and confuse later runs.


In [11]:
exp["tmp"].cleanup()
